# Step 1 — Parse & Chunk MD&A
**Input:** parquet with columns `[accession, ticker, cik, year, sic, naics3, naics_label, mdna_text]`
**Output:** `chunks.jsonl` — one JSON record per chunk with full metadata

In [ ]:
!pip install -q pandas pyarrow tqdm

from google.colab import drive
drive.mount('/content/drive')

import os, json
BASE_DIR   = '/content/drive/MyDrive/mdna_rag'
PARQUET    = f'{BASE_DIR}/mdna_2024_25.parquet'   # ← your input file
CHUNKS_OUT = f'{BASE_DIR}/chunks.jsonl'
os.makedirs(BASE_DIR, exist_ok=True)
print('✅ Ready')

In [ ]:
import re

# ── Section header patterns (order matters — more specific first) ─────────────
SECTION_PATTERNS = [
    ('overview',               r'(?i)^\s*(overview|executive\s+summary|business\s+overview)'),
    ('results_of_operations',  r'(?i)^\s*results\s+of\s+operations'),
    ('revenue',                r'(?i)^\s*(net\s+)?revenue|net\s+sales'),
    ('gross_profit',           r'(?i)^\s*gross\s+(profit|margin)'),
    ('operating_expenses',     r'(?i)^\s*operating\s+exp'),
    ('operating_income',       r'(?i)^\s*operating\s+(income|loss)'),
    ('net_income',             r'(?i)^\s*net\s+(income|loss|earnings)'),
    ('segment_results',        r'(?i)^\s*segment'),
    ('liquidity',              r'(?i)^\s*liquidity\s+and\s+capital'),
    ('cash_flows',             r'(?i)^\s*cash\s+flow'),
    ('critical_accounting',    r'(?i)^\s*critical\s+accounting'),
    ('off_balance_sheet',      r'(?i)^\s*off.balance\s+sheet'),
    ('contractual_obligations',r'(?i)^\s*contractual\s+oblig'),
    ('market_risk',            r'(?i)^\s*(quantitative|market\s+risk)'),
    ('outlook',                r'(?i)^\s*(outlook|forward|guidance)'),
]

def detect_section(line: str) -> str | None:
    for name, pat in SECTION_PATTERNS:
        if re.match(pat, line):
            return name
    return None

print('✅ Section patterns loaded')

In [ ]:
def split_into_sections(text: str) -> list[dict]:
    """Split raw MD&A text into labelled sections."""
    lines   = text.splitlines()
    sections = []
    cur_section = 'preamble'
    cur_lines   = []

    for line in lines:
        detected = detect_section(line)
        if detected:
            if cur_lines:
                sections.append({'section': cur_section,
                                  'text': '\n'.join(cur_lines).strip()})
            cur_section = detected
            cur_lines   = [line]
        else:
            cur_lines.append(line)

    if cur_lines:
        sections.append({'section': cur_section,
                          'text': '\n'.join(cur_lines).strip()})
    return [s for s in sections if len(s['text']) > 80]  # drop trivial sections


def naive_token_count(text: str) -> int:
    return len(text.split())


def chunk_section(section_text: str,
                  max_tokens: int = 512,
                  overlap_tokens: int = 64) -> list[str]:
    """Paragraph-aware chunking within a section."""
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', section_text) if p.strip()]
    chunks, cur_words, overlap_buf = [], [], []

    for para in paragraphs:
        words = para.split()
        if naive_token_count(' '.join(cur_words)) + len(words) > max_tokens:
            if cur_words:
                chunks.append(' '.join(cur_words))
                overlap_buf = cur_words[-overlap_tokens:]  # carry-over
            cur_words = overlap_buf + words
        else:
            cur_words.extend(words)

    if cur_words:
        chunks.append(' '.join(cur_words))
    return chunks

print('✅ Chunking functions defined')

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

df = pd.read_parquet(PARQUET)
print(f'Loaded {len(df):,} filings  |  columns: {list(df.columns)}')
print(df[['ticker','year','naics3','naics_label']].head(3))

In [ ]:
# ── Main chunking loop ────────────────────────────────────────────────────────
MAX_TOKENS    = 512
OVERLAP       = 64
total_chunks  = 0

with open(CHUNKS_OUT, 'w') as fout:
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Chunking filings'):
        mdna_text = str(row.get('mdna_text', '') or '')
        if not mdna_text.strip():
            continue

        sections = split_into_sections(mdna_text)

        for sec in sections:
            sub_chunks = chunk_section(sec['text'], MAX_TOKENS, OVERLAP)
            for idx, chunk_text in enumerate(sub_chunks):
                record = {
                    # ── filing identity ────────────────────────────────────
                    'accession'   : str(row.get('accession', '')),
                    'ticker'      : str(row.get('ticker', '')).upper(),
                    'cik'         : str(row.get('cik', '')),
                    'year'        : int(row.get('year', 0)),

                    # ── industry ───────────────────────────────────────────
                    'sic'         : int(row.get('sic', 0)),
                    'naics3'      : int(row.get('naics3', 0)),
                    'naics_label' : str(row.get('naics_label', '')),

                    # ── chunk content & position ──────────────────────────
                    'section'     : sec['section'],
                    'chunk_index' : idx,
                    'text'        : chunk_text,
                    'token_count' : naive_token_count(chunk_text),
                }
                fout.write(json.dumps(record) + '\n')
                total_chunks += 1

print(f'\n✅ Done — {total_chunks:,} chunks written to {CHUNKS_OUT}')

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────
import json
samples = []
with open(CHUNKS_OUT) as f:
    for i, line in enumerate(f):
        if i >= 3: break
        samples.append(json.loads(line))

for s in samples:
    print(f"[{s['ticker']} {s['year']} | naics3={s['naics3']} | {s['section']}] "
          f"{s['token_count']} tokens")
    print(f"  {s['text'][:120]}...\n")